---
title: "Spark SQL Job Data Analysis"
author: "Your Name"
format: html
embed-resources: true
date: "2020-02-25"
date-format: long
execute:
  echo: true
---

In [6]:
import sys
print(sys.executable)

/home/ubuntu/assignment3/.venv/bin/python


In [7]:
from pyspark.sql import SparkSession

# Start a Spark session
spark = SparkSession.builder.appName("JobPostingsAnalysis").getOrCreate()

# Load the CSV file into a Spark DataFrame
df = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .csv("../data/lightcast_job_postings.csv")

# 3. Register the DataFrame as a temporary SQL table
df.createOrReplaceTempView("jobs")

In [8]:
# Verify the Data

# Display the first five rows
df.show(5)

# Show the schema (column names & data types)
df.printSchema()

+--------------------+-----------------+----------------------+----------+--------+---------+--------+--------------------+--------------------+--------------------+-----------+-------------------+--------------------+--------------------+---------------+----------------+--------+--------------------+-----------+-------------------+----------------+---------------------+-------------+-------------------+-------------+------------------+---------------+--------------------+--------------------+--------------------+-------------+------+-----------+----------------+-------------------+---------+-----------+--------------------+--------------------+-------------+------+--------------+-----+--------------------+-----+----------+---------------+--------------------+---------------+--------------------+------------+--------------------+------------+--------------------+------+--------------------+------+--------------------+------+--------------------+------+--------------------+------+------

### EXAMPLE


In [9]:
# Run a Spark SQL query to count job postings per employment type
job_counts_by_type = spark.sql("""
    SELECT EMPLOYMENT_TYPE_NAME, COUNT(*) AS job_count
    FROM jobs
    GROUP BY EMPLOYMENT_TYPE_NAME
    ORDER BY job_count DESC
""")

# Show the result
job_counts_by_type.show()

+--------------------+---------+
|EMPLOYMENT_TYPE_NAME|job_count|
+--------------------+---------+
|Full-time (> 32 h...|    69176|
|Part-time (â‰¤ 32...|     2298|
|Part-time / full-...|      980|
|                NULL|       44|
+--------------------+---------+



## Query 1: How many job postings we have in the dataset?

In [10]:
spark.sql("SELECT COUNT(*) AS total_job_postings FROM jobs").show()

+------------------+
|total_job_postings|
+------------------+
|             72498|
+------------------+



**Answer:** The dataset contains 72498 job postings in total.

## Query 2: Find the top 5 most common job titles

In [12]:
spark.sql("""
    SELECT TITLE_RAW, COUNT(*) AS job_count
    FROM jobs
    GROUP BY TITLE_RAW
    ORDER BY job_count DESC
    LIMIT 5
""").toPandas()

,TITLE_RAW,job_count
0,Data Analyst,4201
1,Enterprise Architect,808
2,Senior Data Analyst,724
3,Business Intelligence Analyst,686
4,Data Modeler,281


## Query 3: Find the average salary for each employment type

In [13]:
df.columns

['ID',
 'LAST_UPDATED_DATE',
 'LAST_UPDATED_TIMESTAMP',
 'DUPLICATES',
 'POSTED',
 'EXPIRED',
 'DURATION',
 'SOURCE_TYPES',
 'SOURCES',
 'URL',
 'ACTIVE_URLS',
 'ACTIVE_SOURCES_INFO',
 'TITLE_RAW',
 'BODY',
 'MODELED_EXPIRED',
 'MODELED_DURATION',
 'COMPANY',
 'COMPANY_NAME',
 'COMPANY_RAW',
 'COMPANY_IS_STAFFING',
 'EDUCATION_LEVELS',
 'EDUCATION_LEVELS_NAME',
 'MIN_EDULEVELS',
 'MIN_EDULEVELS_NAME',
 'MAX_EDULEVELS',
 'MAX_EDULEVELS_NAME',
 'EMPLOYMENT_TYPE',
 'EMPLOYMENT_TYPE_NAME',
 'MIN_YEARS_EXPERIENCE',
 'MAX_YEARS_EXPERIENCE',
 'IS_INTERNSHIP',
 'SALARY',
 'REMOTE_TYPE',
 'REMOTE_TYPE_NAME',
 'ORIGINAL_PAY_PERIOD',
 'SALARY_TO',
 'SALARY_FROM',
 'LOCATION',
 'CITY',
 'CITY_NAME',
 'COUNTY',
 'COUNTY_NAME',
 'MSA',
 'MSA_NAME',
 'STATE',
 'STATE_NAME',
 'COUNTY_OUTGOING',
 'COUNTY_NAME_OUTGOING',
 'COUNTY_INCOMING',
 'COUNTY_NAME_INCOMING',
 'MSA_OUTGOING',
 'MSA_NAME_OUTGOING',
 'MSA_INCOMING',
 'MSA_NAME_INCOMING',
 'NAICS2',
 'NAICS2_NAME',
 'NAICS3',
 'NAICS3_NAME',
 'NAICS4

In [15]:
spark.sql("""
    SELECT EMPLOYMENT_TYPE, AVG(SALARY) AS avg_salary
    FROM jobs
    WHERE SALARY IS NOT NULL
    GROUP BY EMPLOYMENT_TYPE
    ORDER BY avg_salary DESC
""").show()

+---------------+------------------+
|EMPLOYMENT_TYPE|        avg_salary|
+---------------+------------------+
|              1|118897.55860862407|
|              3| 105621.2423263328|
|              2| 98802.50963391137|
+---------------+------------------+



## Query 4: What five states have the most job postings

In [16]:
spark.sql("""
    SELECT STATE, COUNT(*) AS job_count
    FROM jobs
    GROUP BY STATE
    ORDER BY job_count DESC
    LIMIT 5
""").show()

+-----+---------+
|STATE|job_count|
+-----+---------+
|   48|     8067|
|    6|     7084|
|   12|     3645|
|   51|     3636|
|   17|     3538|
+-----+---------+



## Query 5: Calculate the salary range (max-min) for each job title in a California

In [19]:
spark.sql("""
    SELECT 
        TITLE_RAW,
        MAX(SALARY) - MIN(SALARY) AS salary_range
    FROM jobs
    WHERE STATE_NAME = 'California'
      AND SALARY IS NOT NULL
    GROUP BY TITLE_RAW
    ORDER BY salary_range DESC
""").show()


+--------------------+------------+
|           TITLE_RAW|salary_range|
+--------------------+------------+
|Enterprise Architect|      188442|
|Principal Enterpr...|      170114|
|        Data Analyst|      161760|
| Senior Data Analyst|      155848|
|Senior Enterprise...|      155200|
|   SAP PP Consultant|      149760|
|      Data Analyst 2|      148221|
|Data Engineer, An...|      143160|
|        data analyst|      127920|
|   Lead Data Analyst|      124200|
|Business Intellig...|      124154|
|Data Analyst, Glo...|      110992|
|    Sr. Data Analyst|      109120|
|        Data Modeler|      106124|
| SAP FICO Consultant|      104936|
|  Solution Architect|       98255|
|Senior Data Analy...|       94975|
|Data Analytics En...|       91924|
|Solutions Archite...|       88750|
|Enterprise Applic...|       87714|
+--------------------+------------+
only showing top 20 rows


## Query 6: What top 5 industries have the highest average salaries, and ,more than 100 job postings?

In [22]:
df.columns

['ID',
 'LAST_UPDATED_DATE',
 'LAST_UPDATED_TIMESTAMP',
 'DUPLICATES',
 'POSTED',
 'EXPIRED',
 'DURATION',
 'SOURCE_TYPES',
 'SOURCES',
 'URL',
 'ACTIVE_URLS',
 'ACTIVE_SOURCES_INFO',
 'TITLE_RAW',
 'BODY',
 'MODELED_EXPIRED',
 'MODELED_DURATION',
 'COMPANY',
 'COMPANY_NAME',
 'COMPANY_RAW',
 'COMPANY_IS_STAFFING',
 'EDUCATION_LEVELS',
 'EDUCATION_LEVELS_NAME',
 'MIN_EDULEVELS',
 'MIN_EDULEVELS_NAME',
 'MAX_EDULEVELS',
 'MAX_EDULEVELS_NAME',
 'EMPLOYMENT_TYPE',
 'EMPLOYMENT_TYPE_NAME',
 'MIN_YEARS_EXPERIENCE',
 'MAX_YEARS_EXPERIENCE',
 'IS_INTERNSHIP',
 'SALARY',
 'REMOTE_TYPE',
 'REMOTE_TYPE_NAME',
 'ORIGINAL_PAY_PERIOD',
 'SALARY_TO',
 'SALARY_FROM',
 'LOCATION',
 'CITY',
 'CITY_NAME',
 'COUNTY',
 'COUNTY_NAME',
 'MSA',
 'MSA_NAME',
 'STATE',
 'STATE_NAME',
 'COUNTY_OUTGOING',
 'COUNTY_NAME_OUTGOING',
 'COUNTY_INCOMING',
 'COUNTY_NAME_INCOMING',
 'MSA_OUTGOING',
 'MSA_NAME_OUTGOING',
 'MSA_INCOMING',
 'MSA_NAME_INCOMING',
 'NAICS2',
 'NAICS2_NAME',
 'NAICS3',
 'NAICS3_NAME',
 'NAICS4

In [23]:
spark.sql("""
    SELECT 
        NAICS2_NAME,
        COUNT(*) AS job_count,
        AVG(SALARY) AS avg_salary
    FROM jobs
    WHERE SALARY IS NOT NULL
      AND NAICS2_NAME IS NOT NULL
    GROUP BY NAICS2_NAME
    HAVING COUNT(*) > 100
    ORDER BY avg_salary DESC
    LIMIT 5
""").show()

+--------------------+---------+------------------+
|         NAICS2_NAME|job_count|        avg_salary|
+--------------------+---------+------------------+
|Accommodation and...|      261|145674.50191570882|
|         Information|     2297|140118.73269481934|
|Professional, Sci...|     8981| 132601.5472664514|
|        Retail Trade|      764|124757.09685863875|
|       Manufacturing|     1662|122408.81708784596|
+--------------------+---------+------------------+

